# First-innings cricket score prediction

Train a regression model that estimates a team's final first-innings total from its score at the end of an over. The model uses only match context and information available at that point. Matches are split by `match_id` so deliveries from the same match cannot appear in both training and evaluation.

## 1. Load the datasets and prepare over-by-over snapshots

Each sample is one first-innings state after a completed over. The target is that innings' final total.

In [ ]:
from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd
display = lambda value: print(value.head() if hasattr(value, "head") else value)
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor, VotingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, mean_absolute_error, mean_squared_error, precision_score, r2_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd()
if not (ROOT / 'datasets' / 'matches.csv').exists():
    # Also works when Jupyter is launched from a subfolder.
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'datasets' / 'matches.csv').exists())
DATA_DIR = ROOT / 'datasets'
OUTPUT_DIR = ROOT / 'models'
OUTPUT_DIR.mkdir(exist_ok=True)

matches = pd.read_csv(DATA_DIR / 'matches.csv')
deliveries = pd.read_csv(DATA_DIR / 'deliveries.csv')
print(f'Matches: {len(matches):,}; deliveries: {len(deliveries):,}')
display(matches.head(3))
display(deliveries.head(3))

In [ ]:
# Restrict the task to first innings and retain only fields known by the prediction point.
first = deliveries.loc[deliveries['inning'].eq(1)].copy()
match_context = matches[['id', 'season', 'city', 'venue']].rename(columns={'id': 'match_id'})
first = first.merge(match_context, on='match_id', how='left', validate='many_to_one')
first['season'] = pd.to_numeric(first['season'].astype(str).str.extract(r'(\d{4})')[0], errors='coerce')

# Aggregate by over number as recorded in the IPL ball-by-ball data (0-based).
over_stats = (first.groupby(['match_id', 'inning', 'batting_team', 'bowling_team', 'season', 'city', 'venue', 'over'], dropna=False)
    .agg(runs_in_over=('total_runs', 'sum'), wickets_in_over=('is_wicket', 'sum'), balls_in_over=('ball', 'count'))
    .reset_index().sort_values(['match_id', 'over']))
group_cols = ['match_id', 'inning', 'batting_team', 'bowling_team', 'season', 'city', 'venue']
over_stats['current_score'] = over_stats.groupby(group_cols)['runs_in_over'].cumsum()
over_stats['wickets_lost'] = over_stats.groupby(group_cols)['wickets_in_over'].cumsum().clip(upper=10)
over_stats['overs_elapsed'] = over_stats['over'] + 1
over_stats['balls_faced'] = over_stats.groupby(group_cols)['balls_in_over'].cumsum()
over_stats['run_rate'] = over_stats['current_score'] / over_stats['overs_elapsed']
finals = over_stats.groupby(group_cols, dropna=False)['current_score'].max().rename('final_score').reset_index()
data = over_stats.merge(finals, on=group_cols, how='inner', validate='many_to_one')
# Predict at the end of each completed over; omit the final over because no future score remains.
data = data.loc[data['overs_elapsed'].lt(20) & data['final_score'].gt(0)].copy()

numeric_features = ['season', 'overs_elapsed', 'current_score', 'wickets_lost', 'balls_faced', 'run_rate']
categorical_features = ['batting_team', 'bowling_team', 'city', 'venue']
feature_columns = numeric_features + categorical_features
data[categorical_features] = data[categorical_features].fillna('Unknown').astype(str)
data = data.dropna(subset=['final_score', 'current_score'])
print(f'Training samples: {len(data):,}; first-innings innings: {data.match_id.nunique():,}')
display(data[feature_columns + ['final_score']].head())

## 2. Split by match and train

A grouped holdout gives a more realistic estimate than randomly splitting over snapshots, because all snapshots from a match stay together.

In [ ]:
X = data[feature_columns]
y = data['final_score'].astype(float)
groups = data['match_id']
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

numeric_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scale', StandardScaler())])
categorical_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=2))])
preprocessor = ColumnTransformer([('numeric', numeric_pipe, numeric_features), ('categorical', categorical_pipe, categorical_features)])

model = Pipeline([('preprocess', preprocessor), ('regressor', RandomForestRegressor(n_estimators=25, min_samples_leaf=5, max_features=0.5, n_jobs=1, random_state=42))])
model.fit(X_train, y_train)
predictions = np.clip(model.predict(X_test), 0, None)
metrics = {'MAE': mean_absolute_error(y_test, predictions), 'RMSE': mean_squared_error(y_test, predictions) ** 0.5, 'R2': r2_score(y_test, predictions), 'train_matches': int(groups.iloc[train_idx].nunique()), 'test_matches': int(groups.iloc[test_idx].nunique()), 'test_samples': int(len(test_idx))}
high_score_threshold = 180
actual_high_score = y_test >= high_score_threshold
predicted_high_score = predictions >= high_score_threshold
metrics.update({'high_score_threshold_runs': high_score_threshold,
    'high_score_accuracy': accuracy_score(actual_high_score, predicted_high_score),
    'high_score_precision': precision_score(actual_high_score, predicted_high_score, zero_division=0)})
print(f"High-score definition: final score >= {high_score_threshold} runs")
print(f"High-score accuracy: {metrics['high_score_accuracy']:.1%}")
print(f"High-score precision: {metrics['high_score_precision']:.1%}")
high_score_threshold = 180
actual_high_score = y_test >= high_score_threshold
predicted_high_score = predictions >= high_score_threshold
metrics.update({'high_score_threshold_runs': high_score_threshold, 'high_score_accuracy': accuracy_score(actual_high_score, predicted_high_score), 'high_score_precision': precision_score(actual_high_score, predicted_high_score, zero_division=0)})
print(f"High-score definition: final score >= {high_score_threshold} runs")
print(f"High-score accuracy: {metrics['high_score_accuracy']:.1%}")
print(f"High-score precision: {metrics['high_score_precision']:.1%}")
display(pd.DataFrame([metrics]).round(3))

### High-score classification results (held-out matches)

The regression model predicts a run total, so classification metrics need a score cutoff. Using **180 runs** as the cutoff, the model achieved:

- **Accuracy:** 77.4%
- **Precision:** 69.0%

Here, accuracy is the share of innings correctly classified as below or at least 180. Precision is the share of predicted 180+ innings that actually reached 180. These are threshold-based high-score metrics; MAE and RMSE above evaluate the score estimate itself.

## 3. Review predictions and save the trained model

The held-out predictions below show actual and predicted final totals at different over checkpoints. The saved model expects the same feature fields as the training data.

In [ ]:
results = data.iloc[test_idx][['match_id', 'batting_team', 'overs_elapsed', 'current_score', 'wickets_lost', 'final_score']].copy()
results['predicted_final_score'] = predictions
results['absolute_error'] = (results['final_score'] - results['predicted_final_score']).abs()
display(results.head(15).round(1))

joblib.dump({'pipeline': model, 'feature_columns': feature_columns, 'numeric_features': numeric_features, 'categorical_features': categorical_features}, OUTPUT_DIR / 'first_innings_score_model.joblib')
results.to_csv(OUTPUT_DIR / 'holdout_predictions.csv', index=False)
(OUTPUT_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2))
print(f'Saved model and evaluation files to {OUTPUT_DIR}')

## 4. Try a prediction

Enter a first-innings state after a completed over. The team and venue names should resemble values in the datasets; unseen values are supported.

In [ ]:
def predict_final_score(batting_team, bowling_team, venue, city, season, overs_elapsed, current_score, wickets_lost):
    if not 1 <= overs_elapsed < 20:
        raise ValueError('overs_elapsed must be between 1 and 19 for an in-progress first innings.')
    if not 0 <= wickets_lost <= 10 or current_score < 0:
        raise ValueError('Score and wicket count must be non-negative; wickets_lost must be at most 10.')
    row = pd.DataFrame([{'season': season, 'overs_elapsed': overs_elapsed, 'current_score': current_score,
        'wickets_lost': wickets_lost, 'balls_faced': overs_elapsed * 6,
        'run_rate': current_score / overs_elapsed, 'batting_team': batting_team,
        'bowling_team': bowling_team, 'city': city, 'venue': venue}], columns=feature_columns)
    return round(float(max(current_score, model.predict(row)[0])))

# Example only; change these values to the live match state you want to estimate.
predict_final_score('Chennai Super Kings', 'Mumbai Indians', 'M. A. Chidambaram Stadium', 'Chennai', 2024, 10, 82, 3)

## Notes and limitations
- The prediction updates at over breaks and estimates the final score of the first innings.
- Training data comes from historical matches in the supplied files; teams, rules, venues, or scoring patterns may change over time.
- The simple example helper estimates balls faced as six per completed over. The fitted snapshots use the observed delivery counts, which account for wides and no-balls.